# TF-IDF → TruncatedSVD + Embedding + tabular のMLP/XGBoost実験

同じ時系列foldで次の2実験を比較します。raw TF-IDFはCSRのままTruncatedSVDへ渡し、dense化するのは既定256次元へ圧縮した後だけです。

- S1: TF-IDF SVD + Embedding + tabular/text統計 → MLP
- S2: TF-IDF SVD + Embedding + tabular/text統計 → XGBoost

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from embedding_features import load_embeddings
from ensemble import make_submission
from modeling import (
    S1_TFIDF_SVD_MLP,
    S2_TFIDF_SVD_XGB,
    build_experiment_summary,
    build_oof_predictions,
    default_modeling_config,
    fit_full_tfidf_svd_nonlinear_and_predict_test,
    run_tfidf_svd_nonlinear_experiments,
    save_experiment_outputs,
)
from validation import encode_binary_target, make_time_series_cv

## 1. データ・Embedding cache・実行フラグ

`EMBEDDING_CACHE_DIR`にはTitanまたはCohereで生成済みの、train/test両方が入った同一cache directoryを指定します。列名は確定データに合わせて編集してください。

In [ ]:
TARGET_COL = 'science_tech_decision'
ID_COL = 'project_id'
YEAR_COL = 'project_start_year'
PROJECT_COL = 'project_name'
TEXT_COLS = ['project_name', 'project_objective', 'project_summary', 'current_issues']
NUMERIC_COLS = ['project_start_year', 'project_end_year', 'project_fiscal_year', 'budget']
CATEGORICAL_COLS = ['responsible_ministry']

RUN_CV = False
RUN_FULL_TEST_PREDICTION = False
RUN_S1 = True
RUN_S2 = True

train = pd.read_csv(PROJECT_ROOT / 'input' / 'train.csv')
test = pd.read_csv(PROJECT_ROOT / 'input' / 'test.csv')
train[TARGET_COL] = encode_binary_target(train[TARGET_COL])

EMBEDDING_ROOT = PROJECT_ROOT / 'data' / 'embeddings'
available_caches = sorted(path for path in EMBEDDING_ROOT.glob('*') if path.is_dir())
for position, path in enumerate(available_caches):
    print(position, path)
EMBEDDING_CACHE_DIR = None  # 例: available_caches[0]
print('train:', train.shape, 'test:', test.shape)

In [ ]:
train_embeddings = test_embeddings = None
train_embedding_metadata = test_embedding_metadata = None
if EMBEDDING_CACHE_DIR is not None:
    train_embeddings, train_embedding_metadata = load_embeddings(
        EMBEDDING_CACHE_DIR, 'train', expected_df=train, project_id_col=ID_COL
    )
    test_embeddings, test_embedding_metadata = load_embeddings(
        EMBEDDING_CACHE_DIR, 'test', expected_df=test, project_id_col=ID_COL
    )
    print('train/test embedding:', train_embeddings.shape, test_embeddings.shape)
else:
    print('EMBEDDING_CACHE_DIRを選択してください。')

## 2. 時系列foldとT4設定

TF-IDFとSVDは各fold trainingだけでfitし、validationはtransformだけです。S1/S2は同じTF-IDF・SVDを共有します。SVD次元を確保できないfoldでは自動縮小せずエラーにします。

In [ ]:
folds, cv_diagnostics = make_time_series_cv(
    train, year_col=YEAR_COL, project_col=PROJECT_COL,
    target_col=TARGET_COL, n_valid_years=3,
)
display(cv_diagnostics)

OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'tfidf_svd_nonlinear'
SVD_FEATURE_DIR = PROJECT_ROOT / 'data' / 'csv' / 'tfidf_svd_shared'
CONFIG = default_modeling_config()
CONFIG['text_cols'] = TEXT_COLS
CONFIG['metric'] = 'roc_auc'
CONFIG['tfidf_svd']['n_components'] = 256
CONFIG['tfidf_svd']['n_iter'] = 7
CONFIG['mlp']['device'] = 'cuda'
CONFIG['mlp']['use_amp'] = True
CONFIG['mlp']['early_stop_metric'] = 'auc'
CONFIG['xgboost']['device'] = 'cuda'
CONFIG['xgboost']['tree_method'] = 'hist'
CONFIG['xgboost']['eval_metric'] = 'auc'

embedding_dim = train_embeddings.shape[1] if train_embeddings is not None else 1024
base_dense_dim = CONFIG['tfidf_svd']['n_components'] + embedding_dim
estimated_base_mib = len(train) * base_dense_dim * 4 / 1024**2
print('SVD + Embedding dense dim (tabular除く):', base_dense_dim)
print('full-train base dense memory MiB:', round(estimated_base_mib, 1))
print('OOF output:', OUTPUT_DIR)
print('SVD CSV:', SVD_FEATURE_DIR)

## 3. S1/S2の時系列CV

OOFの古い年度と異常年度は`NaN`のまま保存されます。fold metricsにはseen/unseen AUC、時間、SVD説明分散、実際のSVD次元が入ります。

In [ ]:
svd_results = {}
if RUN_CV:
    if train_embeddings is None:
        raise RuntimeError('EMBEDDING_CACHE_DIRを選択してください。')
    feature_config = {**CONFIG['feature_engineering'], 'text_cols': TEXT_COLS}
    svd_results = run_tfidf_svd_nonlinear_experiments(
        train, folds, TEXT_COLS, train_embeddings, train_embedding_metadata,
        NUMERIC_COLS, CATEGORICAL_COLS,
        run_s1=RUN_S1, run_s2=RUN_S2, target_col=TARGET_COL,
        project_col=PROJECT_COL, project_id_col=ID_COL, year_col=YEAR_COL,
        metric='roc_auc', embedding_scaling=CONFIG['embedding_scaling'],
        tfidf_config=CONFIG['tfidf'], svd_config=CONFIG['tfidf_svd'],
        mlp_config=CONFIG['mlp'], xgboost_config=CONFIG['xgboost'],
        feature_config=feature_config,
        svd_feature_output_dir=SVD_FEATURE_DIR,
        random_state=CONFIG['random_state'],
    )
    oof_predictions = build_oof_predictions(list(svd_results.values()), train.index)
    fold_metrics = pd.concat(
        [result.fold_metrics for result in svd_results.values()], ignore_index=True
    )
    summary = build_experiment_summary(fold_metrics)
    save_experiment_outputs(oof_predictions, fold_metrics, summary, OUTPUT_DIR)
    display(summary)
    display(fold_metrics)
else:
    print('CV is disabled. Set RUN_CV=True when ready.')

## 4. full-train fitと個別submission

trainだけでTF-IDF・SVD・前処理をfitし、testはtransformだけに使います。個別submissionは性能確認用で、最終hill climbing ensembleとは分離します。

In [ ]:
if RUN_FULL_TEST_PREDICTION:
    if train_embeddings is None or test_embeddings is None:
        raise RuntimeError('train/test Embeddingを読み込んでください。')
    selected = []
    if RUN_S1:
        selected.append(S1_TFIDF_SVD_MLP)
    if RUN_S2:
        selected.append(S2_TFIDF_SVD_XGB)
    predictions = fit_full_tfidf_svd_nonlinear_and_predict_test(
        train, test, selected, TEXT_COLS,
        train_embeddings, test_embeddings,
        train_embedding_metadata, test_embedding_metadata,
        NUMERIC_COLS, CATEGORICAL_COLS,
        config=CONFIG, target_col=TARGET_COL, project_id_col=ID_COL,
        svd_feature_output_dir=SVD_FEATURE_DIR / 'full_train',
    )
    prediction_dir = OUTPUT_DIR / 'test_predictions'
    submission_dir = OUTPUT_DIR / 'submissions'
    prediction_dir.mkdir(parents=True, exist_ok=True)
    for experiment, prediction in predictions.items():
        np.save(prediction_dir / f'{experiment}.npy', prediction.astype(np.float32))
        submission = make_submission(
            test, prediction, id_col=ID_COL, prediction_col=TARGET_COL,
            output_path=submission_dir / f'submission_{experiment}.csv',
        )
        assert submission[ID_COL].astype(str).tolist() == test[ID_COL].astype(str).tolist()
        print(experiment, prediction.min(), prediction.max(), len(prediction))
    display(pd.DataFrame({name: value for name, value in predictions.items()}).head())
else:
    print('Full test prediction is disabled. Set RUN_FULL_TEST_PREDICTION=True after CV review.')